# 🦜🔗 LangChain — Pilar 1: Model I/O

> **Versões:** `langchain-core==1.4.0` | `langchain-openai==1.1.10` | Maio/2026

---

## O que é Model I/O?

Model I/O é o **coração do LangChain**. Ele controla tudo que entra e sai de um modelo de linguagem (LLM).

Pense assim:

```
[Prompts]  →  [Model]  →  [Output Parser]
  (entrada)    (cérebro)    (saída estruturada)
```

### Os 3 componentes:

| Componente | O que faz | Analogia |
|---|---|---|
| **Models** | Executa o LLM (OpenAI, Anthropic, etc.) | O cérebro |
| **Prompts** | Formata o texto de entrada | O roteiro do que falar |
| **Output Parsers** | Converte a resposta em dados úteis | O tradutor |

---

## ⚙️ Instalação

Execute a célula abaixo para instalar os pacotes necessários.

In [ ]:
%pip install -qU langchain-core==1.4.0 langchain-openai==1.1.10 pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv('OPENAI_API_KEY'):
    raise EnvironmentError(
        'OPENAI_API_KEY não encontrada.\n'
        'Crie um arquivo .env na raiz do projeto com:\n'
        'OPENAI_API_KEY=sk-...'
    )

print('Ambiente configurado!')

---
## 1. 🧠 Models — O Cérebro

No LangChain, existem dois tipos principais de modelos:

- **LLMs**: recebem uma *string* e retornam uma *string* (modelo legado, menos usado hoje)
- **ChatModels**: recebem uma *lista de mensagens* e retornam uma *mensagem* (padrão atual)

### Tipos de mensagem em um ChatModel:

| Tipo | Quem fala | Uso |
|---|---|---|
| `SystemMessage` | O sistema | Define o comportamento do modelo |
| `HumanMessage` | O usuário | A pergunta ou instrução |
| `AIMessage` | O modelo | A resposta anterior (para histórico) |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Inicializa o modelo
# model='gpt-4o' é o padrão recomendado em 2026
model = ChatOpenAI(model='gpt-4o', temperature=0.7)

# Chamada direta com lista de mensagens
resposta = model.invoke([
    SystemMessage(content='Você é um assistente especialista em Python.'),
    HumanMessage(content='O que é uma list comprehension? Explique em 2 linhas.')
])

print(type(resposta))      # AIMessage
print(resposta.content)    # O texto da resposta

In [ ]:
# Você pode inspecionar os metadados da resposta
print('Modelo usado:', resposta.response_metadata.get('model_name'))
print('Tokens usados:', resposta.usage_metadata)

> **Dica:** `temperature=0` deixa o modelo determinístico (sem criatividade). `temperature=1.0` deixa mais criativo. Para tarefas analíticas, use `0`. Para escrita criativa, use `0.7` a `1.0`.

---
## 2. 📝 Prompts — O Roteiro

Raramente mandamos textos fixos ao modelo. Quase sempre precisamos **preencher variáveis** dinamicamente.

LangChain oferece templates para isso.

### 2.1 PromptTemplate — Para texto simples

In [ ]:
from langchain_core.prompts import PromptTemplate

# Define o template com variáveis entre {chaves}
template = PromptTemplate(
    template='Explique o conceito de {conceito} para um {publico}. Seja direto e use um exemplo prático.',
    input_variables=['conceito', 'publico']
)

# Preenche as variáveis
prompt_preenchido = template.invoke({'conceito': 'recursão', 'publico': 'iniciante em programação'})

print('Tipo:', type(prompt_preenchido))
print('Texto gerado:')
print(prompt_preenchido.text)

### 2.2 ChatPromptTemplate — Para conversas (padrão atual)

O `ChatPromptTemplate` é o mais usado hoje. Ele permite definir múltiplas mensagens com roles diferentes.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Cria o template de chat com system + human
chat_template = ChatPromptTemplate.from_messages([
    ('system', 'Você é um especialista em {area}. Responda sempre em português.'),
    ('human', '{pergunta}')
])

# Preenche as variáveis
mensagens = chat_template.invoke({
    'area': 'machine learning',
    'pergunta': 'Qual a diferença entre overfitting e underfitting?'
})

print('Mensagens geradas:')
for msg in mensagens.messages:
    print(f'  [{msg.type.upper()}]: {msg.content[:80]}...')

### 2.3 MessagesPlaceholder — Para histórico de conversa

Usado quando você quer inserir um **histórico de mensagens** dinamicamente no template.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

template_com_historico = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente útil. Lembre-se do contexto da conversa.'),
    MessagesPlaceholder(variable_name='historico'),  # bloco dinâmico
    ('human', '{pergunta_atual}')
])

# Simula um histórico de conversa
historico_exemplo = [
    HumanMessage(content='Meu nome é Carlos.'),
    AIMessage(content='Prazer, Carlos! Como posso te ajudar?')
]

mensagens = template_com_historico.invoke({
    'historico': historico_exemplo,
    'pergunta_atual': 'Qual é o meu nome?'
})

print(f'Total de mensagens no prompt: {len(mensagens.messages)}')
for msg in mensagens.messages:
    print(f'  [{msg.type.upper()}]: {msg.content}')

---
## 3. 🔧 Output Parsers — O Tradutor

O modelo retorna **texto bruto**. Os Output Parsers **convertem** esse texto em formatos úteis:

| Parser | Saída | Quando usar |
|---|---|---|
| `StrOutputParser` | `str` | Quando você só quer o texto |
| `JsonOutputParser` | `dict` | Quando quer JSON estruturado por um Schema |
| `PydanticOutputParser` | Objeto Pydantic | Quando quer tipagem, validação rígida e Schema complexo |

### 3.1 StrOutputParser — O mais simples

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser_str = StrOutputParser()

# Sem parser: retorna AIMessage
# Com parser: retorna string pura
chain_simples = ChatOpenAI(model='gpt-4o') | parser_str

resultado = chain_simples.invoke('Diga apenas: Olá mundo!')
print(type(resultado))   # <class 'str'>
print(resultado)

### 3.2 JsonOutputParser — Saída em dicionário guiada por Schema

> **Melhoria de código:** Em vez de instruir o JSON manualmente em texto puro, passamos um modelo Pydantic como base para o `JsonOutputParser`. Ele injetará as instruções de formatação dinamicamente, tornando a resposta estruturada muito mais confiável.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Definimos a estrutura que desejamos coletar
class EstruturaPais(BaseModel):
    nome: str = Field(description='Nome do país')
    capital: str = Field(description='Cidade capital')
    populacao_milhoes: float = Field(description='População estimada em milhões de habitantes')
    idioma: str = Field(description='Idioma oficial principal')

# Associamos o modelo Pydantic ao JsonOutputParser
parser_json = JsonOutputParser(pydantic_object=EstruturaPais)

prompt_json = ChatPromptTemplate.from_messages([
    ('system', 'Você é um assistente geográfico. Adote estritamente o formato solicitado.\n{format_instructions}'),
    ('human', 'Me dê informações sobre o país: {pais}.')
])

# Injetamos as instruções automáticas de Schema no prompt utilizando .partial()
chain_json = prompt_json.partial(format_instructions=parser_json.get_format_instructions()) | ChatOpenAI(model='gpt-4o', temperature=0) | parser_json

resultado = chain_json.invoke({'pais': 'Brasil'})

print(type(resultado))   # <class 'dict'>
print(resultado)
print('\nCapital:', resultado.get('capital'))

### 3.3 PydanticOutputParser — Saída tipada e validada

O mais robusto para Engenharia de Software. Converte a resposta diretamente em uma instância de classe de objeto Python validada e tipada.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# 1. Define o schema com Pydantic
class Receita(BaseModel):
    nome: str = Field(description='Nome do prato')
    tempo_minutos: int = Field(description='Tempo de preparo em minutos')
    ingredientes: List[str] = Field(description='Lista de ingredientes principais')
    dificuldade: str = Field(description='Nível: fácil, médio ou difícil')

# 2. Cria o parser
parser_pydantic = PydanticOutputParser(pydantic_object=Receita)

# 3. O parser gera as instruções de formato para o modelo
print('Instruções geradas pelo parser:')
print(parser_pydantic.get_format_instructions()[:300], '...')

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 4. Embute as instruções no prompt
prompt_pydantic = ChatPromptTemplate.from_messages([
    ('system', 'Você é um chef de cozinha. {format_instructions}'),
    ('human', 'Crie uma receita de {prato}.')
])

# 5. Monta a chain completa
chain_pydantic = (
    prompt_pydantic.partial(format_instructions=parser_pydantic.get_format_instructions())
    | ChatOpenAI(model='gpt-4o', temperature=0.5)
    | parser_pydantic
)

receita = chain_pydantic.invoke({'prato': 'bolo de chocolate'})

# Resultado é um objeto Pydantic tipado!
print(type(receita))             # <class 'Receita'>
print('Nome:', receita.nome)
print('Tempo:', receita.tempo_minutos, 'minutos')
print('Dificuldade:', receita.dificuldade)
print('Ingredientes:', receita.ingredientes)

---
## 4. 🔗 Pipeline Completo com LCEL

O operador `|` (pipe) é a forma moderna de conectar componentes no LangChain.

```
prompt | model | output_parser
```

Isso cria um **RunnableSequence** — uma cadeia que executa cada etapa em ordem.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# 1. Prompt
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Você é um professor de {materia}. Seja didático e use exemplos.'),
    ('human', 'Explique {topico} em no máximo 3 parágrafos.')
])

# 2. Modelo
modelo = ChatOpenAI(model='gpt-4o', temperature=0.3)

# 3. Parser
parser = StrOutputParser()

# 4. Chain: conecta tudo com |
chain = prompt | modelo | parser

print('Tipo da chain:', type(chain).__name__)  # RunnableSequence

# 5. Executa
resposta = chain.invoke({
    'materia': 'Data Science',
    'topico': 'o que é uma regressão linear'
})

print('\n' + '='*60)
print(resposta)

In [ ]:
# Streaming — recebe a resposta token por token
print('Resposta em streaming:\n')

for chunk in chain.stream({'materia': 'Python', 'topico': 'decoradores'}):
    print(chunk, end='', flush=True)

print('\n\n[streaming concluído]')

---
## 📚 Resumo

| Conceito | Classe principal | Para que serve |
|---|---|---|
| **ChatModel** | `ChatOpenAI` | Executa o LLM via API |
| **Mensagens** | `SystemMessage`, `HumanMessage` | Estruturam a conversa |
| **Prompt simples** | `PromptTemplate` | Template com variáveis para string |
| **Prompt de chat** | `ChatPromptTemplate` | Template com múltiplas mensagens |
| **Histórico** | `MessagesPlaceholder` | Insere mensagens dinâmicas no prompt |
| **Parser texto** | `StrOutputParser` | Extrai só o texto da AIMessage |
| **Parser JSON** | `JsonOutputParser` | Converte para dicionário Python via Schema |
| **Parser tipado** | `PydanticOutputParser` | Converte para objeto Pydantic validado |
| **Pipeline LCEL** | operador `\|` | Conecta componentes em sequência |

> **Próximo passo:** `02_data_connection_rag.ipynb` — Como conectar o modelo a dados externos com RAG.